# Data Processing Pipeline

We start with the full LendingClub dataset, which contains over 2.26 million loan records and 145 columns. Since we can only measure how much principal was actually recovered after a loan has ended, we first remove all loans that are still in progress and keep only the ones that have reached a final status — `Fully Paid`, `Charged Off`, or `Default` — leaving about 1.3 million rows. We then define our prediction target as `principal_recovery_ratio = total_rec_prncp / loan_amnt`, which tells us what fraction of the original loan amount was paid back (ranging from 0 to 1, with a mean around 0.86). To make sure the model cannot "cheat" by seeing the answer, we carefully exclude all outcome-related columns (such as total payments, recoveries, and outstanding principal) from the input features, and only keep 28 variables that would have been known at the time the loan was issued — things like loan amount, interest rate, credit grade, annual income, employment length, debt-to-income ratio, and so on.

Next, we clean and standardize these raw features so they are ready for modeling. Many columns arrive as messy strings — for example, `"36 months"` is converted to the number `36`, and `"10+ years"` of employment becomes `10`. Date fields like the borrower's earliest credit line are turned into a more useful numeric feature: `years_since_earliest_cr`, which measures how long the borrower has had credit. The letter-based credit grades are converted to ordered numbers (`A`=1 through `G`=7, and similarly for sub-grades). We also run an exploratory data analysis (EDA) to understand the data before modeling: we look at how loan amounts and incomes are distributed, how recovery rates differ across credit grades and loan terms, and which features are correlated with each other.

Finally, we prepare the data for machine learning. The dataset is randomly split into three non-overlapping parts — 70% for training (912,546 rows), 15% for validation (195,545 rows), and 15% for testing (195,547 rows). We then build a preprocessing pipeline using scikit-learn's `ColumnTransformer`: for the 23 numeric columns, missing values are filled with the median and then scaled to have zero mean and unit variance; for the 6 categorical columns, missing values are filled with the most common category and then one-hot encoded into binary indicators. This pipeline is fit only on the training set to prevent information leakage, and then applied to all three sets, producing a final feature matrix with 101 dimensions that is ready to be fed into our models.